# Day 5: Cross-Validation and Evaluation Metrics from Scratch

**Goal:** Implement K-Fold Cross-Validation manually in NumPy, then build
the core evaluation metrics used in classification.

In [51]:
import numpy as np
import pandas as pd
from wrapped_models.decision_tree import DecisionTree
from wrapped_models.xgboost_classifier import XGBoostClassifier
from wrapped_models.random_forest import RandomForest
from wrapped_models.logistic_regression import LogisticRegression

## K-Fold Cross-Validation

A single train/validation split is fragile — depending on which samples
land in validation, your accuracy estimate can be too optimistic or too
pessimistic. K-Fold fixes this by repeating the evaluation K times.

The dataset is divided into K equal **folds**. In each iteration one fold
becomes the validation set and the remaining K-1 folds are used for
training. This repeats until every fold has been the validation set
exactly once.

The final score is the **average metric across all K folds**:

$$\text{CV Score} = \frac{1}{K} \sum_{k=1}^{K} \text{metric}(y_k,\ \hat{y}_k)$$

For example with K=5 and 100 samples:

| Fold | Train indices | Validation indices |
|------|--------------|-------------------|
| 1    | 20–99        | 0–19              |
| 2    | 0–19, 40–99  | 20–39             |
| 3    | 0–39, 60–99  | 40–59             |
| 4    | 0–59, 80–99  | 60–79             |
| 5    | 0–79         | 80–99             |

Two things K-Fold gives you that a single split cannot:
- **Lower variance** — the score is averaged over K different validation
  sets, so one unlucky split cannot skew the result
- **Full data usage** — every sample is used for both training and
  validation across the K rounds

In [52]:
def kfold_split(n: int, k: int, shuffle: bool = True) -> list[tuple[list[int], list[int]]]:
    indices = list(range(n))
    if shuffle:
        np.random.shuffle(indices)

    fold_size = n // k
    remainder = n % k   

    begin = 0
    splits = []

    for _ in range(k):
        end = begin + fold_size if remainder <= 0 else begin + fold_size + 1
        remainder -= 1

        validation_indices = indices[begin : end]
        training_indices = indices[:begin] + indices[end:]
        
        splits.append((training_indices, validation_indices))
        begin = end
    
    return splits

## Executing Cross-Validation

With the split logic in place, we need a function to manage the training and validation process across the folds. This function will take a model, the dataset, and the number of folds, then return the accuracy for each fold.

In [53]:
def cross_validate(model, X: pd.DataFrame, y: np.ndarray, k: int, shuffle: bool) -> float:
    splits = kfold_split(n=len(X), k=k, shuffle=shuffle)
    accuracies = []

    for training_indices, validation_indices in splits:
        # train model on train indices and validate on val ones
        model.fit(X.iloc[training_indices], y[training_indices])
        predictions = np.array(model.predict(X.iloc[validation_indices])) == y[validation_indices]
        
        accuracy = np.mean(predictions)
        accuracies.append(accuracy)

    # ret CV score which is the mean across accuracies
    return float(np.mean(accuracies))

## The Confusion Matrix

Every metric we compute comes from four numbers. For each prediction
our model makes, exactly one of these outcomes occurs:

|                     | Predicted Positive  | Predicted Negative  |
|---------------------|---------------------|---------------------|
| **Actual Positive** | TP (True Positive)  | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative)  |

- **TP** — predicted positive, was positive (correct)
- **TN** — predicted negative, was negative (correct)
- **FP** — predicted positive, was negative (false alarm)
- **FN** — predicted negative, was positive (missed it)

In [54]:
def confusion_matrix(y_actual: np.ndarray, y_pred: np.ndarray) -> tuple[int, int, int, int]:
    TP = np.sum((y_pred == y_actual) & (y_actual == 1))    # model predicted true and was right
    FP = np.sum((y_pred != y_actual) & (y_pred == 1))      # model predicted true and was not right
    TN = np.sum((y_pred == y_actual) & (y_actual == 0))    # model predicted false and was right
    FN = np.sum((y_pred != y_actual) & (y_pred == 0))      # model predicted false and was not right

    return TP, FP, TN, FN

## Accuracy

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

Of all predictions, what fraction was correct?

**The problem:** if 95% of samples are class 0, a model that always
predicts 0 scores 95% accuracy while being completely useless. This is
why accuracy alone is never enough on imbalanced data.

In [55]:
def accuracy(y_actual: np.ndarray, y_pred: np.ndarray) -> float:
    tp, fp, tn, fn = confusion_matrix(y_actual, y_pred)

    return (tp + tn) / (tp + tn + fp + fn)

## Precision

$$\text{Precision} = \frac{TP}{TP + FP}$$

Of everything the model called positive, how many actually were?

**Use when false positives are costly.** A spam filter that wrongly
blocks real emails has low precision — it is too aggressive. You want
precision high so that when the model says positive, you can trust it.

In [56]:
def precision(y_actual: np.ndarray, y_pred: np.ndarray) -> float:
    tp, fp, _, _ = confusion_matrix(y_actual, y_pred)

    return tp / (tp + fp)

## Recall

$$\text{Recall} = \frac{TP}{TP + FN}$$

Of all actual positives, how many did the model catch?

**Use when false negatives are costly.** A cancer screening that misses
real cases has low recall — it is too conservative. In medicine, missing
a sick patient is far worse than a false alarm, so recall is often the
primary metric.

Note that precision and recall pull against each other. Predicting
positive on everything gives recall = 1.0 but destroys precision.

In [57]:
def recall(y_actual: np.ndarray, y_pred: np.ndarray) -> float:
    tp, _, _, fn = confusion_matrix(y_actual, y_pred)

    return tp / (tp + fn)

## F1 Score

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

A single number that balances precision and recall. It uses harmonic
mean instead of a regular average, which means **both must be high for
F1 to be high**. A model with precision = 1.0 and recall = 0.1 gets
F1 = 0.18, not 0.55.

Use F1 when classes are imbalanced and both false positives and false
negatives matter.

In [58]:
def f1(y_actual: np.ndarray, y_pred: np.ndarray) -> float:
    curr_recall = recall(y_actual, y_pred)
    curr_precision = precision(y_actual, y_pred)

    return 2 * curr_precision * curr_recall / (curr_recall + curr_precision)

## Comparing Models with Cross-Validation

With our metrics in place, we can now run a full comparison across all
four models. For each model we run K-Fold CV and report not just
accuracy but all four metrics, computed on the combined out-of-fold
predictions.

A high accuracy with low recall on the positive class is a signal the
model is taking the easy route — predicting negative most of the time.

In [59]:
# prepare the dummy dataset
data = {
    'Age': [22, 25, 47, 35, 14, 50, 28, 19, 60, 38],
    'Sex': ['male', 'female', 'female', 'male', 'male', 'female', 'male', 'female', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 1, 1, 0, 1, 0, 1]
}
df_dummy = pd.DataFrame(data)

# map categorical features to numeric
df_dummy['Sex'] = df_dummy['Sex'].map({'male': 1, 'female': 0})

# force float typing to prevent NumPy casting errors
X_dummy = df_dummy[['Age', 'Sex']].astype(float)
y_dummy = df_dummy['Survived'].to_numpy()

# initialize all models
models = {
    "Logistic Regression": LogisticRegression(learning_rate=0.01, epochs=100),
    "Decision Tree": DecisionTree(max_depth=3, min_samples=2, random_subspace=False),
    "Random Forest": RandomForest(n_trees=5),
    "XGBoost Classifier": XGBoostClassifier(n_estimators=5, learning_rate=0.3, max_depth=2)
}

# evaluate each model using out-of-fold predictions
print("--- Model Comparison (Out-of-Fold Metrics) ---")
for name, model in models.items():
    splits = kfold_split(n=len(X_dummy), k=3, shuffle=True)
    
    all_y_actual = []
    all_y_pred = []
    
    # collect predictions across all folds
    for train_indices, val_indices in splits:
        model.fit(X_dummy.iloc[train_indices], y_dummy[train_indices])
        predictions = model.predict(X_dummy.iloc[val_indices])
        
        all_y_actual.extend(y_dummy[val_indices])
        all_y_pred.extend(predictions)
        
    all_y_actual = np.array(all_y_actual)
    all_y_pred = np.array(all_y_pred)
    
    # calculate metrics
    model_accuracy = accuracy(all_y_actual, all_y_pred)
    model_precision = precision(all_y_actual, all_y_pred)
    model_recall = recall(all_y_actual, all_y_pred)
    model_f1 = f1(all_y_actual, all_y_pred)
    
    print(f"\n{name}:")
    print(f"  Accuracy : {model_accuracy:.4f}")
    print(f"  Precision: {model_precision:.4f}")
    print(f"  Recall   : {model_recall:.4f}")
    print(f"  F1 Score : {model_f1:.4f}")

--- Model Comparison (Out-of-Fold Metrics) ---

Logistic Regression:
  Accuracy : 0.4000
  Precision: 0.5000
  Recall   : 0.3333
  F1 Score : 0.4000

Decision Tree:
  Accuracy : 0.5000
  Precision: 0.5556
  Recall   : 0.8333
  F1 Score : 0.6667

Random Forest:
  Accuracy : 0.6000
  Precision: 0.6000
  Recall   : 1.0000
  F1 Score : 0.7500

XGBoost Classifier:
  Accuracy : 0.9000
  Precision: 1.0000
  Recall   : 0.8333
  F1 Score : 0.9091
